# Flux — hitra veljavnostna preverba (delni batchi)

Preveri, ali je scFEA flux **uporaben kot modaliteta**, PREDEN cakas na vseh 14 batchov.
Bere ze narejene `flux_*.csv` z Drive (nic ne racuna znova, samo bere).

**Zazeni v LOCENEM runtime** (drugi notebook `01` se lahko vzporedno vrti).

Sodba temelji na 4 testih:
1. brez NaN / ne prazno (vhod OK?)
2. flux varira MED celicami (nosi informacijo?)
3. porazdelitev ni degenerirana
4. flux korelira z RNA (bioloska smiselnost)


## 0. Mount Drive

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Veljavnostna preverba

Ce `SKUPNA SODBA` = UPORABEN -> pusti `01` do konca.
Ce PROBLEM -> ustavi `01`, ne zapravljaj ur (preveri vhod: geni/prazne celice).

In [ ]:
# ============================================================================
# HITRA VELJAVNOST FLUKSA (na delnih batchih, npr. 4/14)
# Preveri ce je flux UPORABEN kot modaliteta PREDEN cakas na vse batche.
# Zazeni v Colabu (kjer je Drive mountan). Nic ne spreminja, samo bere.
# ============================================================================
import os, glob, pickle
import numpy as np, pandas as pd

DATA = '/content/drive/MyDrive/Diploma/data/processed'
FLUX_DIR = os.path.join(DATA, 'flux_batches')

# 1) nalozi vse ZE NAREJENE batche
batch_files = sorted(glob.glob(os.path.join(FLUX_DIR, 'flux_*.csv')),
                     key=lambda p: int(p.split('flux_')[1].split('.csv')[0]))
print(f'Najdenih batchov: {len(batch_files)} -> {[os.path.basename(f) for f in batch_files]}')
assert batch_files, 'NI batchov! Preveri pot.'

flux = pd.concat([pd.read_csv(f, index_col=0) for f in batch_files], axis=0)
V = flux.values.astype(np.float64)
print(f'Flux (delni): {flux.shape}  (celice x moduli)')
print(f'Moduli: {list(flux.columns[:3])} ... {list(flux.columns[-2:])}')

print('\n=== TEST 1: brez NaN / ne prazno ===')
n_nan = int(np.isnan(V).sum())
frac0 = (V == 0).mean()
print(f'NaN: {n_nan}  (mora biti 0)')
print(f'delez tocnih nicel: {frac0:.3f}  (nekaj nicel OK; >0.9 = sumljivo)')
print(f'min / max / mean: {V.min():.4f} / {V.max():.4f} / {V.mean():.4f}')
t1 = (n_nan == 0) and (frac0 < 0.9)
print('-> TEST 1:', 'OK' if t1 else 'PADEL (flux prazen/NaN -> vhod zafuran)')

print('\n=== TEST 2: variabilnost MED celicami (nosi informacijo?) ===')
# flux mora biti razlicen med celicami, sicer je neuporaben kot modaliteta
per_cell_std = V.std(axis=1)                     # razprsenost modulov znotraj celice
per_module_std = V.std(axis=0)                   # razprsenost celic znotraj modula
n_dead_modules = int((per_module_std < 1e-9).sum())
print(f'std med celicami (povpr. po modulih): {per_module_std.mean():.4f}')
print(f'"mrtvih" modulov (isti flux v vseh celicah): {n_dead_modules}/{V.shape[1]}')
print(f'celic z vsemi enakimi moduli (std=0): {int((per_cell_std < 1e-9).sum())}/{V.shape[0]}')
t2 = per_module_std.mean() > 1e-4 and n_dead_modules < V.shape[1] * 0.5
print('-> TEST 2:', 'OK' if t2 else 'PADEL (flux ne varira -> ne nosi informacije)')

print('\n=== TEST 3: porazdelitev razumna (ne degenerirana) ===')
# koliko modulov ima smiselno varianco; ali je nekaj ekstremnih outlierjev
cv = per_module_std / (V.mean(axis=0) + 1e-9)    # koeficient variacije na modul
print(f'modulov s CV>0.1 (variabilni): {int((cv > 0.1).sum())}/{V.shape[1]}')
print(f'razpon mean po modulih: {V.mean(axis=0).min():.4f} .. {V.mean(axis=0).max():.4f}')
p1, p50, p99 = np.percentile(V, [1, 50, 99])
print(f'percentili 1/50/99: {p1:.4f} / {p50:.4f} / {p99:.4f}')
t3 = int((cv > 0.1).sum()) > V.shape[1] * 0.2
print('-> TEST 3:', 'OK' if t3 else 'OPOZORILO (malo variabilnih modulov)')

# 4) BIOLOSKA SMISELNOST: flux vs RNA na istih celicah (najmocnejsi test)
print('\n=== TEST 4: flux korelira z RNA? (biolopska smiselnost) ===')
try:
    with open(os.path.join(DATA, 'data_rna_counts.pkl'), 'rb') as f:
        counts = pickle.load(f)
    with open(os.path.join(DATA, 'gene_names.pkl'), 'rb') as f:
        gene_names = [str(g) for g in pickle.load(f)]
    n_have = V.shape[0]                            # samo prve celice (delni flux)
    from scipy.sparse import issparse
    C = counts[:n_have]
    total_rna = np.asarray(C.sum(axis=1)).ravel() if issparse(C) else C.sum(axis=1)
    total_flux = V.sum(axis=1)
    # celice z vec RNA naj imajo vec skupnega fluksa (groba, a smiselna povezava)
    mask = total_rna > 0
    r = np.corrcoef(np.log1p(total_rna[mask]), total_flux[mask])[0, 1]
    print(f'korelacija log(skupni RNA) vs skupni flux: r = {r:.3f}  (na {mask.sum()} celicah)')
    print('-> pozitiven r (>0.2) = smiselno (vec RNA -> vec metabol. aktivnosti)')
except Exception as e:
    print('preskocheno (RNA ni na voljo ali napaka):', str(e)[:80])

print('\n' + '='*60)
verdict = t1 and t2
print('SKUPNA SODBA:', 'FLUX JE UPORABEN -> nadaljuj z ostalimi batchi' if verdict
      else 'PROBLEM -> ustavi, preveri vhod (ne zapravljaj casa)')
print('='*60)


## 2. (neobvezno) Vizualna kontrola

Ce sodba OK, si lahko pogledas porazdelitev fluksa cez module.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
# a) povprecen flux na modul (kateri moduli so aktivni)
mean_per_mod = V.mean(axis=0)
ax[0].bar(range(len(mean_per_mod)), sorted(mean_per_mod, reverse=True))
ax[0].set_title('Povprecen flux na modul (padajoce)')
ax[0].set_xlabel('modul (rang)'); ax[0].set_ylabel('povpr. flux')
# b) porazdelitev variabilnosti (std) po modulih
ax[1].hist(per_module_std, bins=40)
ax[1].set_title('Variabilnost (std) med celicami, po modulih')
ax[1].set_xlabel('std'); ax[1].set_ylabel('st. modulov')
plt.tight_layout(); plt.show()
print('Ce levi graf: nekaj modulov visokih + rep -> normalno.')
print('Ce desni graf: vecina modulov ima std>0 -> flux nosi info.')